In [22]:
import ollama
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from typing import List
import tiktoken as tkn
import re

In [6]:
client = QdrantClient(url="http://localhost:6333")

In [2]:
def embed_text(text):
    embeding = ollama.embeddings(model='nomic-embed-text', prompt=text)
    return embeding

In [17]:
def chunk_prompt(prompt: str, chunk_size: int = 2000, overlap: int = 50) -> List[str]:
    """Chunks a long text into smaller segments based on token count."""
    print("Begin chunking the page")
    print(f"Total length of the page: {len(prompt)}")

    # Note: Using tiktoken for gpt-3.5-turbo is an approximation.
    # Tokenization will vary slightly for different Ollama models.
    # encoding = tkn.encoding_for_model("gpt-3.5-turbo")
    sentences = re.split(r"([.?!])", prompt)
    # Group sentences and their terminators back together
    sentences = ["".join(i) for i in zip(sentences[0::2], sentences[1::2])]

    n_tokens = [len(sentence) for sentence in sentences]
    chunks, tokens_so_far, chunk = [], 0, []

    for sentence, token in zip(sentences, n_tokens):
        if tokens_so_far + token > chunk_size:
            chunks.append("".join(chunk))
            if overlap > 0:
                # A simple overlap strategy; more sophisticated methods could be used.
                overlap_sentences = [s for s in chunk if len(s) > 0][-overlap:]
                chunk = overlap_sentences
                tokens_so_far = sum(len(s) for s in overlap_sentences)
            else:
                chunk, tokens_so_far = [], 0

        chunk.append(sentence)
        tokens_so_far += token

    if chunk:
        chunks.append("".join(chunk))

    print(f"Total number of chunks: {len(chunks)}")
    return chunks


In [18]:
long_text = """

Lorem ipsum dolor sit amet, consectetur adipiscing elit. Nullam finibus, ante consectetur maximus tempus, dolor mi semper sem, ut malesuada leo arcu ut justo. Vivamus sed tellus a dolor egestas finibus. Praesent varius sapien ut quam cursus egestas. Curabitur tincidunt at neque nec fringilla. Pellentesque nec purus nec libero vestibulum pellentesque ac eu velit. Nullam venenatis urna sit amet ante fringilla fermentum. Quisque a nunc eleifend sem rhoncus maximus ac ut velit. Integer interdum orci mauris, at pretium magna commodo eu. Duis pretium sollicitudin neque quis vehicula. Proin nec sodales nulla. Maecenas semper est ut odio malesuada blandit. Donec in metus varius, hendrerit lorem sed, tempus metus.

Fusce aliquet metus in magna malesuada iaculis. Mauris varius euismod augue in hendrerit. Sed id arcu eget massa euismod posuere sed non leo. Donec feugiat ligula vitae magna faucibus vestibulum. Donec a turpis at massa sollicitudin ultrices. In hac habitasse platea dictumst. Nulla hendrerit ipsum sed nisl tempus tempor.

Ut blandit enim a iaculis tincidunt. Fusce luctus ante at enim finibus, vel efficitur nunc viverra. Donec non elit nisi. Sed maximus ultricies suscipit. Vivamus molestie porttitor dignissim. Mauris placerat fringilla ligula, non ullamcorper massa interdum eget. Vestibulum dapibus neque semper nulla rhoncus malesuada. Sed velit nunc, facilisis non purus et, consectetur maximus orci. Donec in imperdiet tortor. Nunc est urna, euismod sit amet aliquam eu, cursus eget neque. Proin elementum feugiat leo, ac eleifend nulla rutrum a. Cras tincidunt condimentum ante, id blandit ex pellentesque vel. Curabitur rutrum consectetur arcu, at elementum velit mattis non. Nulla ut interdum diam. Nam sit amet luctus arcu. Orci varius natoque penatibus et magnis dis parturient montes, nascetur ridiculus mus.

Mauris nunc eros, facilisis ac augue vitae, auctor posuere ipsum. Sed vel enim facilisis, molestie diam ut, facilisis urna. Suspendisse suscipit maximus leo, non blandit velit egestas non. Quisque ut elit nulla. Phasellus consequat purus vel tincidunt congue. Morbi placerat mauris semper lectus sollicitudin eleifend. Aliquam viverra semper elit id tempor. Praesent aliquet sollicitudin nunc vitae varius. Donec cursus nunc sit amet finibus pulvinar. Aliquam eget commodo metus. Phasellus maximus, ante in rutrum pharetra, est erat cursus elit, condimentum facilisis mi purus at est. Praesent rutrum justo at augue viverra, ut aliquet erat dapibus. Pellentesque eu aliquet erat.

Curabitur sollicitudin vulputate lorem et vestibulum. Morbi sed odio non eros accumsan gravida ac semper metus. Nam dui magna, accumsan quis pharetra auctor, consectetur eget nunc. Sed eget sapien vitae nibh rutrum faucibus eget vestibulum purus. Integer viverra dolor a justo pulvinar, sodales dapibus lectus varius. Aliquam erat volutpat. Praesent convallis tortor ac diam volutpat, eu venenatis augue pharetra. Vestibulum congue ligula nec fringilla facilisis. Sed congue, tellus nec accumsan ornare, sapien erat interdum turpis, non condimentum sem eros in mauris. Integer vel volutpat ex. Pellentesque sit amet massa non eros egestas dapibus semper nec libero. Vestibulum sed diam luctus, sollicitudin erat eu, porttitor dui. Mauris ultrices euismod nulla, sed tincidunt enim iaculis ac. Nullam dignissim bibendum tempus. Curabitur est eros, interdum sit amet massa nec, varius ultricies tortor. Nunc tincidunt augue in dui lobortis, in luctus elit blandit.

Phasellus elementum cursus ligula sed lobortis. Pellentesque habitant morbi tristique senectus et netus et malesuada fames ac turpis egestas. Duis venenatis magna elit, non condimentum nunc elementum id. Quisque suscipit est justo, eu malesuada quam sagittis vel. Nullam viverra fringilla ligula, vitae malesuada erat consectetur vel. Aenean quam leo, dignissim ac risus sit amet, pellentesque facilisis purus. Proin ultricies metus libero, id interdum ligula commodo id. Nullam et commodo nunc.

Morbi eu nibh rhoncus, tempus velit quis, dapibus elit. Sed fermentum bibendum faucibus. Etiam non mollis sem, sed iaculis justo. Sed blandit ac eros et euismod. Duis tempus augue nisl. Integer nec condimentum dolor, a sollicitudin lorem. Nulla eu placerat purus. Vivamus auctor id lectus id ornare. Cras tempor nisl ac turpis gravida, id commodo velit consectetur. Etiam molestie at sem ultricies ornare. Morbi pellentesque mi tincidunt, tincidunt erat a, hendrerit orci.

Mauris a urna tincidunt, vestibulum orci quis, tempor est. Sed imperdiet sagittis mi, sed volutpat nunc mollis nec. Vivamus congue tellus sed lacus pulvinar, sed varius elit eleifend. Nullam rhoncus risus est, quis faucibus dui maximus a. Phasellus sit amet egestas nulla, vitae venenatis purus. Integer scelerisque, massa vitae tempus vehicula, neque purus euismod metus, a condimentum est magna vitae nibh. Aliquam erat volutpat. Vestibulum ut justo non urna viverra lacinia id id massa. Proin ultricies ut felis vitae interdum. Phasellus nulla nisl, finibus vitae fringilla vel, dictum at dolor. Pellentesque quis diam et lectus cursus egestas nec quis ante. Fusce magna ligula, ornare eget blandit ut, rutrum eu dolor. Duis porttitor neque eget nulla vestibulum, sed lacinia dui dignissim. Nunc maximus ligula quis finibus aliquet.

Integer rhoncus maximus auctor. Aenean condimentum lacus vel dapibus laoreet. Duis vitae tortor vitae libero varius elementum. Suspendisse eget orci justo. Quisque et sodales tellus. Vivamus aliquam tempor ante in condimentum. Nunc vitae erat quis orci pulvinar congue. Quisque nec mauris fermentum, tristique nisl vel, bibendum magna. Nullam tristique velit in tempus auctor. Aenean sed eleifend odio, bibendum hendrerit ante. Nam gravida viverra imperdiet. Quisque sodales mi nec lectus tristique laoreet. Duis ut elit cursus, ullamcorper libero ac, aliquam sem. Etiam et turpis interdum, egestas elit sit amet, lacinia felis.

Ut mattis mauris et lacus porttitor egestas. Aenean quis justo in justo blandit vehicula. Duis convallis tristique mi. Nulla pharetra elementum augue, ac condimentum quam tempor non. Curabitur euismod mi in tincidunt accumsan. Nam luctus risus nec ullamcorper semper. Sed ut aliquam urna. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia curae; Mauris ipsum justo, lacinia a ante ac, dapibus rhoncus lectus. Nunc fringilla enim velit, eget scelerisque nisl auctor nec.

Mauris at dignissim tellus. Phasellus molestie urna et quam ornare pellentesque. Nam luctus aliquet leo, at fringilla massa volutpat et. Nunc et nunc magna. Phasellus sodales accumsan orci et hendrerit. Nam leo justo, porta sit amet nunc id, tristique varius nulla. Praesent risus arcu, lacinia a odio faucibus, pretium iaculis orci. Aenean ut sem mollis, mollis nisi. """

In [21]:
chunk_prompt(prompt=long_text, chunk_size=200, overlap=0)

Begin chunking the page
Total length of the page: 6797
Total number of chunks: 41


['\n\nLorem ipsum dolor sit amet, consectetur adipiscing elit. Nullam finibus, ante consectetur maximus tempus, dolor mi semper sem, ut malesuada leo arcu ut justo.',
 ' Vivamus sed tellus a dolor egestas finibus. Praesent varius sapien ut quam cursus egestas. Curabitur tincidunt at neque nec fringilla.',
 ' Pellentesque nec purus nec libero vestibulum pellentesque ac eu velit. Nullam venenatis urna sit amet ante fringilla fermentum. Quisque a nunc eleifend sem rhoncus maximus ac ut velit.',
 ' Integer interdum orci mauris, at pretium magna commodo eu. Duis pretium sollicitudin neque quis vehicula. Proin nec sodales nulla. Maecenas semper est ut odio malesuada blandit.',
 ' Donec in metus varius, hendrerit lorem sed, tempus metus.\n\nFusce aliquet metus in magna malesuada iaculis. Mauris varius euismod augue in hendrerit.',
 ' Sed id arcu eget massa euismod posuere sed non leo. Donec feugiat ligula vitae magna faucibus vestibulum. Donec a turpis at massa sollicitudin ultrices. In hac h

In [4]:
text = "this is a test"

print(len(embed_text(text).embedding))

768


In [25]:
import PyPDF2
from tqdm import tqdm

# client.create_collection(
#     collection_name="submarine",
#     vectors_config=VectorParams(size=768, distance=Distance.DOT),
# )

from qdrant_client.models import PointStruct

# sky_embedding = ollama.embeddings(model='nomic-embed-text', prompt='The sky is blue because of rayleigh scattering')


# operation_info = client.upsert(
#     collection_name="test_collection",
#     wait=True,
#     points=[
#         PointStruct(id=1, vector=sky_embedding.embedding, payload={"path": "../sops/projecta/sop1.docx", "page": 2})
#     ],
# )

# ground_embedding = ollama.embeddings(model='nomic-embed-text', prompt='The ground is blue because of something else')

# operation_info = client.upsert(
#     collection_name="test_collection",
#     wait=True,
#     points=[
#         PointStruct(id=2, vector=ground_embedding.embedding, payload={"path": "../sops/projecta/sop2.docx", "page": 3})
#     ],
# )

OLLAMA_EMBEDDING_MODEL = "nomic-embed-text"

pdf_file_path = '/home/sam/Documents/Projects/RAG/RAG-Project/data/history of submarines.pdf'
with open(pdf_file_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        embeddings = []

        for page_num, page in enumerate(tqdm(reader.pages, desc="Processing Pages")):
            text = page.extract_text()
            if not text or not text.strip():
                continue

            page_chunks = chunk_prompt(text, chunk_size=500, overlap=5)

            for index, chunk in enumerate(page_chunks):
                # Generate embedding for each chunk using Ollama
                response = ollama.embeddings(
                    model=OLLAMA_EMBEDDING_MODEL,
                    prompt=chunk
                )
                operation_info = client.upsert(
                collection_name="submarine",
                wait=True,
                points=[
                    PointStruct(id=page_num * 1000 + index, vector=response.embedding, payload={"page": page_num, "context" : chunk})
                ],
)

Processing Pages:   0%|          | 0/7 [00:00<?, ?it/s]

Begin chunking the page
Total length of the page: 49
Total number of chunks: 0
Begin chunking the page
Total length of the page: 2798
Total number of chunks: 19


Processing Pages:  29%|██▊       | 2/7 [00:00<00:01,  2.50it/s]

Begin chunking the page
Total length of the page: 3184
Total number of chunks: 28


Processing Pages:  43%|████▎     | 3/7 [00:01<00:02,  1.53it/s]

...

Processing Pages:  71%|███████▏  | 5/7 [00:03<00:01,  1.35it/s]

Begin chunking the page
Total length of the page: 3600
Total number of chunks: 22


Processing Pages:  86%|████████▌ | 6/7 [00:04<00:00,  1.27it/s]

Begin chunking the page
Total length of the page: 2957
Total number of chunks: 18


Processing Pages: 100%|██████████| 7/7 [00:05<00:00,  1.39it/s]


In [28]:
query_embedded = ollama.embeddings(model='nomic-embed-text', prompt='first example')

print(query_embedded)

search_result = client.query_points(
    collection_name="submarine",
    query=query_embedded.embedding,
    with_payload=True,
    limit=3
).points

print(search_result)

embedding=[0.7361487746238708, 0.4315928816795349, -4.02818489074707, -0.7311030030250549, 0.7843666672706604, -0.2746233344078064, 0.4561638832092285, 0.044737037271261215, -0.6202268600463867, 1.134584665298462, -0.8953120708465576, 0.5603638887405396, 2.0516128540039062, 1.6912248134613037, -0.31248772144317627, -0.42081397771835327, 0.1814379096031189, -2.094316005706787, -0.3714136481285095, -0.34502533078193665, -0.47733184695243835, 0.628529965877533, -1.1061019897460938, -0.17457619309425354, 1.9411362409591675, 0.20098450779914856, -0.03008274734020233, 0.15983843803405762, -1.2673869132995605, -0.673585057258606, -0.4168320894241333, 0.36357027292251587, -0.1820017695426941, -1.024334192276001, -0.7097951173782349, -0.8698967099189758, 0.9289443492889404, 0.46690240502357483, 0.8807857036590576, -0.4340031147003174, -0.17369484901428223, -1.004547119140625, 0.46360570192337036, 0.2621656656265259, 0.41260483860969543, -1.3072490692138672, 1.0727390050888062, 0.414196908473968